# Download NASA HLS-VI NDVI Manifests with Cloud Cover ≤ 10%

This notebook creates **one `.txt` manifest per province**:

```text
<province>_ndvi_s3.txt
```

Each line is an S3 link to an HLS-VI NDVI `.tif` file whose corresponding base HLS Sentinel-2 granule has cloud coverage `<= 10`.

## 0. Optional package installation

In [ ]:
# Uncomment only if needed.
# %pip install requests pandas geopandas shapely tqdm

## 1. Configuration

Start with `TEST_PROVINCES = ["shanghai"]`.  
After verifying that the Shanghai manifest is non-empty, set:

```python
TEST_PROVINCES = None
```

to run all provinces.

In [2]:
from pathlib import Path
import re
import time

import requests
import geopandas as gpd
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Local project paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:/Users/26540/Desktop/学习/Uchicago/final-project-greenness-infrastructure-happiness")

BOUNDARY_FILE = PROJECT_ROOT / "boundary_data" / "geoBoundaries-CHN-ADM1.geojson"

# Only .txt files named <province>_ndvi_s3.txt will be written here.
MANIFEST_OUT_DIR = PROJECT_ROOT / "manifests"
MANIFEST_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# CMR / HLS settings
# ---------------------------------------------------------------------
CMR_GRANULE_URL = "https://cmr.earthdata.nasa.gov/search/granules.umm_json"

# Query the base HLS Sentinel-2 surface-reflectance collection.
# Then derive the matching HLS-VI NDVI S3 path.
BASE_SHORT_NAME = "HLSS30"
BASE_VERSION = "2.0"
BASE_COLLECTION_CONCEPT_ID = "C2021957295-LPCLOUD"
PROVIDER = "LPCLOUD"

YEAR = 2018
START_DATE = f"{YEAR}-01-01T00:00:00Z"
END_DATE = f"{YEAR}-12-31T23:59:59Z"

MAX_CLOUD_COVER = 10.0

PAGE_SIZE = 200
REQUEST_SLEEP_SECONDS = 0.1

# Keep this False because CMR-side cloud filtering can over-filter.
# We query broadly and filter cloud coverage in Python.
USE_CMR_CLOUD_FILTER = False

# Test first. Set to None after the test works.
TEST_PROVINCES = ["shanghai"]
# TEST_PROVINCES = None

# Exclude provinces you do not need in the final CGSS sample.
EXCLUDE_PROVINCES = {"xinjiang"}

print("Boundary file:", BOUNDARY_FILE)
print("Manifest output directory:", MANIFEST_OUT_DIR)

Boundary file: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\boundary_data\geoBoundaries-CHN-ADM1.geojson
Manifest output directory: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\manifests


## 2. Load province boundaries

In [3]:
if not BOUNDARY_FILE.exists():
    raise FileNotFoundError(f"Boundary file not found: {BOUNDARY_FILE}")

gdf = gpd.read_file(BOUNDARY_FILE)

print("Boundary columns:", gdf.columns.tolist())
print("Boundary CRS:", gdf.crs)

# CMR bounding_box requires lon/lat in WGS84.
if gdf.crs is not None and str(gdf.crs).lower() not in ["epsg:4326", "wgs84"]:
    gdf = gdf.to_crs("EPSG:4326")

name_candidates = ["shapeName", "shapeName_en", "NAME_1", "name", "province", "Province"]
name_col = None
for c in name_candidates:
    if c in gdf.columns:
        name_col = c
        break

if name_col is None:
    raise ValueError(f"Cannot find province-name column. Columns are: {gdf.columns.tolist()}")

print("Using province-name column:", name_col)
print("Number of boundary rows:", len(gdf))

Boundary columns: ['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType', 'geometry']
Boundary CRS: EPSG:4326
Using province-name column: shapeName
Number of boundary rows: 34


## 3. Normalize province names

In [4]:
def normalize_province_name(x):
    s = str(x).strip().lower()

    replacements = {
        "beijing municipality": "beijing",
        "tianjin municipality": "tianjin",
        "shanghai municipality": "shanghai",
        "chongqing municipality": "chongqing",

        "inner mongolia autonomous region": "inner_mongolia",
        "nei mongol autonomous region": "inner_mongolia",
        "neimenggu": "inner_mongolia",

        "guangxi zhuang autonomous region": "guangxi",
        "ningxia hui autonomous region": "ningxia",

        "xinjiang uyghur autonomous region": "xinjiang",
        "xinjiang uighur autonomous region": "xinjiang",

        "tibet autonomous region": "tibet",
        "xizang autonomous region": "tibet",

        "shaanxi province": "shaanxi",
        "sha-nxi province": "shaanxi",
        "shanxi province": "shanxi",

        # Your boundary file has used this label before.
        "guangzhou province": "guangdong",
    }

    if s in replacements:
        return replacements[s]

    s = s.replace(" province", "")
    s = s.replace(" municipality", "")
    s = s.replace(" autonomous region", "")
    s = s.replace(" zhuang", "")
    s = s.replace(" hui", "")
    s = s.replace(" uyghur", "")
    s = s.replace(" uighur", "")
    s = s.replace("-", "_")
    s = re.sub(r"\s+", "_", s)
    return s

gdf["province_key"] = gdf[name_col].apply(normalize_province_name)

gdf_run = gdf.copy()

if EXCLUDE_PROVINCES:
    gdf_run = gdf_run[~gdf_run["province_key"].isin(EXCLUDE_PROVINCES)].copy()

if TEST_PROVINCES is not None:
    wanted = {normalize_province_name(x) for x in TEST_PROVINCES}
    gdf_run = gdf_run[gdf_run["province_key"].isin(wanted)].copy()

if len(gdf_run) == 0:
    raise ValueError("No province selected. Check TEST_PROVINCES and EXCLUDE_PROVINCES.")

print("Selected provinces:")
print(sorted(gdf_run["province_key"].unique()))
print("Number selected:", len(gdf_run))

Selected provinces:
['shanghai']
Number selected: 1


## 4. Helper functions

In [5]:
def bbox_to_cmr_string(bounds):
    minx, miny, maxx, maxy = bounds
    return f"{minx},{miny},{maxx},{maxy}"


def flatten_values(value):
    if value is None:
        return []
    if isinstance(value, list):
        out = []
        for x in value:
            out.extend(flatten_values(x))
        return out
    if isinstance(value, dict):
        out = []
        for k in ["Value", "Values", "value", "values"]:
            if k in value:
                out.extend(flatten_values(value[k]))
        return out
    return [value]


def get_cloud_cover(umm):
    # Extract cloud cover from UMM metadata.
    # Returns None if cloud-cover metadata is unavailable.
    for key in ["CloudCover", "CloudCoverage", "cloud_cover", "cloud_coverage"]:
        if key in umm and umm[key] is not None:
            for v in flatten_values(umm[key]):
                try:
                    return float(v)
                except Exception:
                    pass

    for attr in umm.get("AdditionalAttributes", []) or []:
        name = str(attr.get("Name", "")).strip().lower()
        name_norm = re.sub(r"[^a-z0-9]+", "_", name).strip("_")

        if name_norm in {
            "cloud_cover",
            "cloud_coverage",
            "cloudcover",
            "cloudcoverage",
            "percentage_of_cloud_cover",
        }:
            vals = []
            for k in ["Values", "Value"]:
                vals.extend(flatten_values(attr.get(k)))
            for v in vals:
                try:
                    return float(v)
                except Exception:
                    pass

    return None


def get_granule_ur(entry):
    umm = entry.get("umm", {}) or {}
    meta = entry.get("meta", {}) or {}
    return (
        umm.get("GranuleUR")
        or meta.get("native-id")
        or meta.get("native_id")
        or meta.get("concept-id")
    )


def extract_base_hls_id(granule_ur):
    # Extract base HLS Sentinel-2 granule ID.
    # Example: HLS.S30.T51SUR.2018301T022811.v2.0
    if not granule_ur:
        return None

    m = re.search(
        r"HLS\.S30\.T\d{2}[A-Z]{3}\.\d{7}T\d{6}\.v2\.0",
        str(granule_ur)
    )
    if m:
        return m.group(0)

    return None


def base_hls_to_vi_ndvi_s3(base_hls_id):
    # Convert base HLS granule ID to HLS-VI NDVI protected S3 path.
    if not base_hls_id:
        return None

    vi_id = base_hls_id.replace("HLS.S30.", "HLS-VI.S30.")
    return f"s3://lp-prod-protected/HLSS30_VI.020/{vi_id}/{vi_id}.NDVI.tif"


def parse_granule_entry(entry):
    umm = entry.get("umm", {}) or {}
    granule_ur = get_granule_ur(entry)
    base_hls_id = extract_base_hls_id(granule_ur)

    return {
        "granule_ur": granule_ur,
        "base_hls_id": base_hls_id,
        "cloud_cover": get_cloud_cover(umm),
        "ndvi_s3": base_hls_to_vi_ndvi_s3(base_hls_id),
    }

## 5. Query CMR

In [6]:
def build_query_modes():
    # Try collection concept ID first. If it returns zero, fall back to
    # short_name/version and short_name-only queries.
    return [
        {
            "label": "concept_id",
            "params": {"collection_concept_id": BASE_COLLECTION_CONCEPT_ID},
        },
        {
            "label": "short_name_version",
            "params": {
                "provider": PROVIDER,
                "short_name": BASE_SHORT_NAME,
                "version": BASE_VERSION,
            },
        },
        {
            "label": "short_name_only",
            "params": {
                "provider": PROVIDER,
                "short_name": BASE_SHORT_NAME,
            },
        },
    ]


def query_cmr_granules_for_bbox_with_mode(bbox_string, mode):
    page_num = 1
    entries = []

    while True:
        params = {
            **mode["params"],
            "temporal": f"{START_DATE},{END_DATE}",
            "bounding_box": bbox_string,
            "page_size": PAGE_SIZE,
            "page_num": page_num,
            "sort_key[]": "start_date",
        }

        if USE_CMR_CLOUD_FILTER:
            params["cloud_cover"] = f"0,{MAX_CLOUD_COVER}"

        r = requests.get(CMR_GRANULE_URL, params=params, timeout=120)

        if page_num == 1:
            print(f"  mode={mode['label']}, status={r.status_code}, CMR-Hits={r.headers.get('CMR-Hits')}")
            print(f"  url={r.url}")

        r.raise_for_status()
        data = r.json()
        page_entries = data.get("items", [])

        if not page_entries:
            break

        entries.extend(page_entries)

        if len(page_entries) < PAGE_SIZE:
            break

        page_num += 1
        time.sleep(REQUEST_SLEEP_SECONDS)

    return entries


def query_cmr_granules_for_bbox(bbox_string):
    for mode in build_query_modes():
        entries = query_cmr_granules_for_bbox_with_mode(bbox_string, mode)
        if entries:
            print(f"  selected_mode={mode['label']}, returned={len(entries)}")
            return entries

    return []

## 6. Write `<province>_ndvi_s3.txt`

This cell is the only cell that writes output files.  
It writes only `.txt` files named exactly:

```text
<province>_ndvi_s3.txt
```

In [7]:
for _, row in tqdm(gdf_run.iterrows(), total=len(gdf_run)):
    province = row["province_key"]
    bbox_string = bbox_to_cmr_string(row.geometry.bounds)

    print(f"\n=== {province} ===")
    print("bbox:", bbox_string)

    entries = query_cmr_granules_for_bbox(bbox_string)
    parsed = [parse_granule_entry(e) for e in entries]

    ndvi_s3_links = sorted({
        p["ndvi_s3"]
        for p in parsed
        if p["cloud_cover"] is not None
        and p["cloud_cover"] <= MAX_CLOUD_COVER
        and p["ndvi_s3"] is not None
    })

    out_txt = MANIFEST_OUT_DIR / f"{province}_ndvi_s3.txt"
    out_txt.write_text(
        "\n".join(ndvi_s3_links) + ("\n" if ndvi_s3_links else ""),
        encoding="utf-8"
    )

    n_cloud_metadata = sum(p["cloud_cover"] is not None for p in parsed)
    print(f"granules_returned: {len(parsed)}")
    print(f"granules_with_cloud_metadata: {n_cloud_metadata}")
    print(f"cloud_cover_le_10_ndvi_s3_links: {len(ndvi_s3_links)}")
    print(f"wrote: {out_txt}")

  0%|          | 0/1 [00:00<?, ?it/s]


=== shanghai ===
bbox: 121.0018690170001,30.710089537000044,122.0441424600001,31.836917734000053
  mode=concept_id, status=200, CMR-Hits=527
  url=https://cmr.earthdata.nasa.gov/search/granules.umm_json?collection_concept_id=C2021957295-LPCLOUD&temporal=2018-01-01T00%3A00%3A00Z%2C2018-12-31T23%3A59%3A59Z&bounding_box=121.0018690170001%2C30.710089537000044%2C122.0441424600001%2C31.836917734000053&page_size=200&page_num=1&sort_key%5B%5D=start_date


100%|██████████| 1/1 [00:04<00:00,  4.50s/it]

  selected_mode=concept_id, returned=527
granules_returned: 527
granules_with_cloud_metadata: 527
cloud_cover_le_10_ndvi_s3_links: 87
wrote: C:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\manifests\shanghai_ndvi_s3.txt


## 7. Check generated TXT manifests

In [8]:
txt_files = sorted(MANIFEST_OUT_DIR.glob("*_ndvi_s3.txt"))

print("TXT manifests written:")
for p in txt_files:
    n_lines = sum(1 for line in p.read_text(encoding="utf-8").splitlines() if line.strip())
    print(f"{p.name}: {n_lines} links")

TXT manifests written:
anhui_ndvi_s3.txt: 154 links
beijing_ndvi_s3.txt: 73 links
chongqing_ndvi_s3.txt: 143 links
fujian_ndvi_s3.txt: 167 links
gansu_ndvi_s3.txt: 2523 links
guangdong_ndvi_s3.txt: 330 links
guangxi_ndvi_s3.txt: 301 links
guizhou_ndvi_s3.txt: 204 links
hebei_ndvi_s3.txt: 100 links
heilongjiang_ndvi_s3.txt: 1359 links
henan_ndvi_s3.txt: 198 links
hubei_ndvi_s3.txt: 445 links
hunan_ndvi_s3.txt: 256 links
jiangsu_ndvi_s3.txt: 147 links
jiangxi_ndvi_s3.txt: 288 links
jilin_ndvi_s3.txt: 434 links
liaoning_ndvi_s3.txt: 100 links
neimenggu_ndvi_s3.txt: 3267 links
ningxia_ndvi_s3.txt: 438 links
qinghai_ndvi_s3.txt: 923 links
sha-nxi_ndvi_s3.txt: 476 links
shandong_ndvi_s3.txt: 216 links
shanghai_ndvi_s3.txt: 87 links
shanxi_ndvi_s3.txt: 1128 links
shenzhen_ndvi_s3.txt: 37 links
sichuan_ndvi_s3.txt: 702 links
tianjing_ndvi_s3.txt: 38 links
xinjiang_ndvi_s3.txt: 1775 links
yunnan_ndvi_s3.txt: 978 links
zhejiang_ndvi_s3.txt: 201 links
